# RALM — thí nghiệm dừng unlearning trên Kaggle (GPU T4 ×2)

Repo: https://github.com/tiensinh2/Stopping_for_MU · Hướng dẫn chi tiết: `KAGGLE.md` · Các điểm khác kịch bản: `CHANGES.md`

**Trước khi chạy:** Settings → Accelerator **GPU T4 x2**, Internet **On**.

**Thứ tự:** `0` thiết lập → `1` smoke test + go/no-go → `2` (tuỳ chọn) lr-scan → `3` chạy chính → `4` kết quả → `5` đóng gói.

- Mọi lệnh **idempotent**: job đã xong bị bỏ qua, nên session chết thì chạy lại cell là tiếp tục (miễn `/kaggle/working/ralm_out` còn; xem mục 5).
- **Chưa có kết quả nào được kiểm chứng trên CIFAR-10 thật.** Nếu cell nào lỗi, hãy gửi log, đừng tự đổi recipe (§18 kịch bản).

## 0. Thiết lập

In [ ]:
import os, sys, json, glob, shutil, time, zipfile

# ---- cấu hình (sửa ở đây)
REPO_URL = "https://github.com/tiensinh2/Stopping_for_MU.git"
REPO_DIR = "/kaggle/working/ralm_repo"
OUT      = "/kaggle/working/ralm_out"      # ngoài repo: git pull không đụng tới
PROFILE  = "kaggle"                         # kaggle: 25 bước/snapshot, 1000 bước | paper: 10 bước, 2000 bước

# smoke test (mục 1)
SM_SEEDS, SM_FKEYS, SM_METHODS = "0", "random_rho0.01", "ga ngp scrub"
# chạy chính (mục 3)
SEEDS   = "0 1 2"
FKEYS   = "random_rho0.1 random_rho0.01"
METHODS = "ga ngp scrub salun tarun"
RETRAIN_SEEDS = 1                           # kịch bản §9 cho tới 5; tăng khi còn compute

In [ ]:
%cd /kaggle/working
# repo private: lưu token GitHub (quyền Contents: read-only) vào Add-ons -> Secrets với tên GH_TOKEN. Repo public thì bỏ qua được.
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://%s@" % UserSecretsClient().get_secret("GH_TOKEN"))
except Exception:
    pass

if os.path.isdir(REPO_DIR + "/.git"):
    !cd {REPO_DIR} && git pull {url} main
else:
    !git clone {url} {REPO_DIR}
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL}
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
import torch
NGPU = torch.cuda.device_count()
print("GPU:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU >= 1, "Không thấy GPU: bật Accelerator = GPU T4 x2 rồi Restart session"
!python -m ralm.run info
!python -m pytest tests -x -q

## 1. Smoke test (bắt buộc — Gate 0.5, §12)

Một seed, random ρ=0.01, GA + NG+ + SCRUB. **GATE θ0:** `base` tự dừng nếu test acc < 93% (§3.1) — khi đó gửi log, đừng đổi recipe.
Ước tính (chưa đo): base ~0.5–1 giờ trên T4.

In [ ]:
t0 = time.time()
!python -m ralm.run base    --profile {PROFILE} --out {OUT} --seeds {SM_SEEDS}
!python -m ralm.run launch retrain --gpus {NGPU} --profile {PROFILE} --out {OUT} --seeds {SM_SEEDS} --fkeys {SM_FKEYS} --retrain_seeds {RETRAIN_SEEDS}
!python -m ralm.run launch traj    --gpus {NGPU} --profile {PROFILE} --out {OUT} --seeds {SM_SEEDS} --fkeys {SM_FKEYS} --methods {SM_METHODS}
print("xong sau %.0f phút" % ((time.time() - t0) / 60))

In [ ]:
!python -m ralm.run aggregate --profile {PROFILE} --out {OUT} --seeds {SM_SEEDS} --fkeys {SM_FKEYS} --methods {SM_METHODS} --tag smoke

### Go / no-go (§12)
- **GO** nếu `ralm-dv` có |k̂ − k*| ≤ 100 bước, **hoặc** regret nhỏ hơn cả `fixed-200` lẫn trung vị `random-k`.
- **NO-GO** nếu `ralm-dv` luôn chọn k̂ = k_min, hoặc corr(J_DV, G_logit_F) < 0 (J thấp = tệ).
- NO-GO → dừng, báo lại; không chạy mục 3.

In [ ]:
import numpy as np
print("%-34s %6s %6s %9s %9s %9s %9s %8s  %s" % ("run", "k_hat", "k*", "reg_dv", "reg_fix200", "reg_rand", "reg_retain", "corrJG", "verdict"))
for f in sorted(glob.glob(OUT + "/results/runs/*.json")):
    o = json.load(open(f, encoding="utf-8"))
    R = {r["rule"]: r for r in o["rules"]}
    dv = R["ralm-dv"]
    rnd = float(np.median([r["regret"] for k, r in R.items() if k.startswith("random-k")]))
    corr = o["go_nogo"]["corr_J_G"]
    fx = R["fixed-200"] if "fixed-200" in R else min((r for k, r in R.items() if k.startswith("fixed-") and k != "fixed-last"),
                                                     key=lambda r: abs(r["k_hat"] - 200))
    near = abs(dv["k_hat"] - o["k_star"]) <= 100
    beats = dv["regret"] < fx["regret"] and dv["regret"] < rnd
    nogo = bool(dv["at_kmin"]) or (corr == corr and corr < 0)      # corr==corr: loại NaN
    verdict = "NO-GO" if nogo else ("GO" if (near or beats) else "không rõ (không GO)")
    print("%-34s %6d %6d %9.4f %9.4f %9.4f %9.4f %8.3f  %s" % (o["run_id"][:34], dv["k_hat"], o["k_star"], dv["regret"],
          fx["regret"], rnd, R["retain-only"]["regret"], corr, verdict))

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(OUT + "/results/rules_table_smoke.md", encoding="utf-8").read()))
for p in sorted(glob.glob(OUT + "/results/figs/regret_curve_*.png")):
    display(Image(p))

## 2. (Tuỳ chọn, nên làm) lr-scan cho SalUn / Tarun / SCRUB

Siêu tham số các phương pháp này **chưa được hiệu chỉnh trên dữ liệu thật** (xem `CHANGES.md` mục 1). Theo §5.1: quét lr trên **1 seed**,
chọn bằng tiêu chí **L_V(step 200) ≤ 1.2·L_V(0)** (chỉ dùng V, không dùng test hay retrain). Chạy xong thì sửa **một lần** `METHOD_HP` trong
`ralm/config.py`, ghi `CHANGES.md`, commit/push, rồi `git pull` ở mục 0 và giữ nguyên cho mọi seed (§18.5).
`run_id` không chứa lr nên mỗi (method, lr) dùng một thư mục `--out` riêng.

In [ ]:
SCAN_METHODS = ["salun", "tarun", "scrub"]
SCAN_LRS     = [1e-3, 5e-3, 2e-2]
SCAN_SEED, SCAN_FKEY = 0, "random_rho0.1"
SCAN_STEPS = 200

# base cho seed quét (nếu mục 1 chưa train seed này)
!python -m ralm.run base --profile {PROFILE} --out {OUT} --seeds {SCAN_SEED}

scan = []
for m in SCAN_METHODS:
    for lr in SCAN_LRS:
        d = "/kaggle/working/lrscan/%s_%g" % (m, lr)
        for sub in ("ckpt", "data"):
            if not os.path.isdir("%s/%s" % (d, sub)):
                shutil.copytree("%s/%s" % (OUT, sub), "%s/%s" % (d, sub), dirs_exist_ok=True)
        !python -m ralm.run traj --profile {PROFILE} --out {d} --seeds {SCAN_SEED} --fkeys {SCAN_FKEY} --methods {m} --lr {lr} --n_steps {SCAN_STEPS}
        rows = [json.loads(l) for l in open(glob.glob(d + "/logs/traj_%s_*seed%d.jsonl" % (m, SCAN_SEED))[0], encoding="utf-8")]
        r0, rN = rows[0], rows[-1]
        scan.append((m, lr, rN["L_V"] / r0["L_V"], rN["L_F"] / r0["L_F"], rN["A_V"], rN["A_F"]))

print("%-8s %-8s %-10s %-10s %-7s %-7s %s" % ("method", "lr", "LV/LV0", "LF/LF0", "A_V", "A_F", "đạt (LV<=1.2 LV0)"))
for m, lr, a, b, av, af in scan:
    print("%-8s %-8g %-10.3f %-10.3f %-7.3f %-7.3f %s" % (m, lr, a, b, av, af, "OK" if a <= 1.2 else "-"))

## 3. Chạy chính (Track A, §0B.3)

`launch` mở một tiến trình cho mỗi GPU, chia job theo `i % NGPU`. Ước tính tổng cỡ 6–10 giờ GPU cho 3 seed × 2 ρ × 5 phương pháp
(**chưa đo** — hãy xem thời gian thật ở smoke rồi quyết định giảm seed). Chạy từng cell riêng; nếu session hết giờ, chạy lại cell, job xong sẽ bị bỏ qua.

In [ ]:
t0 = time.time()
!python -m ralm.run launch base --gpus {NGPU} --profile {PROFILE} --out {OUT} --seeds {SEEDS}
print("base xong sau %.0f phút" % ((time.time() - t0) / 60))

In [ ]:
t0 = time.time()
!python -m ralm.run launch retrain --gpus {NGPU} --profile {PROFILE} --out {OUT} --seeds {SEEDS} --fkeys {FKEYS} --retrain_seeds {RETRAIN_SEEDS}
print("retrain xong sau %.0f phút" % ((time.time() - t0) / 60))

In [ ]:
t0 = time.time()
!python -m ralm.run launch traj --gpus {NGPU} --profile {PROFILE} --out {OUT} --seeds {SEEDS} --fkeys {FKEYS} --methods {METHODS}
print("trajectory xong sau %.0f phút" % ((time.time() - t0) / 60))
errs = glob.glob(OUT + "/logs/ERROR_*.txt")
print("job lỗi:", errs if errs else "không có")

## 4. Kết quả

In [ ]:
!python -m ralm.run aggregate --profile {PROFILE} --out {OUT} --seeds {SEEDS} --fkeys {FKEYS} --methods {METHODS} --tag trackA

In [ ]:
import pandas as pd
from IPython.display import Markdown, Image, display
R = OUT + "/results"
display(Markdown(open(R + "/rules_table_trackA.md", encoding="utf-8").read()))
for p in sorted(glob.glob(R + "/figs/box_*.png") + glob.glob(R + "/figs/scatter_*.png")):
    display(Image(p))

In [ ]:
# H4 band, privacy proxy, ma trận anchor T0-T10 (re-analysis, 0 GPU)
R = OUT + "/results"
band = pd.read_csv(R + "/band_trackA.csv")
print("== H4 band: coverage (>=80% cần cho H4), path_width kỳ vọng 5-30 snapshot ==")
print(band.groupby(["method", "fkey"]).agg(coverage=("coverage", "mean"), path_width=("path_width", "median"),
      G_in_mean=("G_in_mean", "median"), G_out_mean=("G_out_mean", "median")).round(4))
print("\n== MIA AUC (logit-gap, F vs test; 0.5 = không rò rỉ) ==")
print(pd.read_csv(R + "/mia_trackA.csv").drop(columns="run_id").describe().loc[["mean", "50%"]].round(3))
print("\n== Anchor thắng T0 (§0A.6.2: -20% highloss VÀ classwise, không tệ >10% random) ==")
tm = pd.read_csv(R + "/tmatrix_trackA.csv")
print(tm[tm.WIN] if tm.WIN.any() else "không anchor nào thắng (hoặc chưa chạy đủ highloss/classwise)")

In [ ]:
# số liệu đối chiếu điều kiện thành công/bác bỏ §14 cho ralm-dv
s = pd.read_csv(R + "/summary_trackA.csv")
s["rule_g"] = s["rule"].str.replace(r"random-k\d", "random-k", regex=True)
piv = s.pivot_table(index=["method", "fkey"], columns="rule_g", values="regret", aggfunc="median")
cols = [c for c in ("ralm-dv", "ralm-d", "retain-only", "val-only", "fixed-200", "fixed-last", "random-k") if c in piv]
print("median regret (nhỏ hơn tốt hơn). Báo cáo luôn kèm IQR trong rules_table (§18.4).")
print(piv[cols].round(4))
d = s[s.rule == "ralm-dv"]
print("\n%% run ralm-dv chọn k_hat = k_min: %.0f%%  (>=50%% là bác bỏ theo §14.2)" % (100 * d.at_kmin.mean()))

## 5. Đóng gói

`/kaggle/working` bị xoá khi session mới bắt đầu. Hãy **Save Version** (Save & Run All hoặc Quick Save) để giữ output,
hoặc tải file zip dưới đây. Muốn tiếp tục ở session khác: đưa `ralm_out/ckpt` + `ralm_out/data` + `ralm_out/logs` vào một Kaggle Dataset
rồi copy về `/kaggle/working/ralm_out` trước khi chạy lại các cell (job đã xong sẽ được bỏ qua).

In [ ]:
zp = "/kaggle/working/ralm_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            p = os.path.join(root, fn)
            rel = os.path.relpath(p, OUT)
            keep_small = rel.startswith("results") or (rel.startswith("logs") and not fn.endswith(".npz")) or rel.startswith("data" + os.sep + "forget_manifest") or rel.startswith("data" + os.sep + "base_stats")
            if keep_small:
                z.write(p, rel)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 2**20), "(không gồm ckpt/*.pth và *.npz nặng)")